# Week 1 · Tutorial 3 — Reading and describing data, and splitting it

**Covers videos 1.7 – 1.8:** a two-minute routine for any new dataset · training, validation and test sets.

**What you will do**
1. Run the *shape – dtypes – describe – isna* routine on hourly data from a **biogas combined heat and power plant**
   and fix what it reveals.
2. Split a time series **by time**, and a table of transformers **by group** and **stratified**, and see
   what a careless random split does.

You work with `pandas` only. No modelling is needed; the point is to *prepare* the data honestly.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/blasern/data-tutorials/main/data/"   # where the data files live
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. The two-minute routine

`biogas_chp.csv` holds hourly readings from a biogas CHP plant for the year 2023. **One row is one hour.**

| column | meaning |
|---|---|
| `timestamp` | hour (UTC) |
| `engine_status` | `running` / `stopped` |
| `electric_kw`, `heat_kw` | average electrical and thermal output during the hour |
| `methane_pct` | methane content of the biogas |
| `digester_temp_c`, `ambient_c` | temperatures |

The routine: **shape → dtypes → describe → isna.**

In [ ]:
chp = pd.read_csv(DATA_URL + "biogas_chp.csv")
expected_rows = 365 * 24
print("shape:", chp.shape, "| expected rows for 2023:", expected_rows, "| difference:", len(chp) - expected_rows)
print()
print(chp.dtypes)

In [ ]:
chp.describe().round(1)

In [ ]:
chp.isna().sum()

Things to notice: more rows than hours in a year; the `timestamp` is text; `ambient_c` has a minimum of **-999**;
`electric_kw` has a maximum far above the installed capacity of **500 kW**; there are missing values.

### ✏️ Your turn 1

**Duplicates.** The count does not add up because some hours appear twice.

1. Fill in the `TODO`s: count the fully duplicated rows with `chp.duplicated().sum()` and remove them with
   `chp.drop_duplicates()`.
2. Convert `timestamp` to a proper date-time type and check that the number of rows now equals the number of hours in 2023
   (the cell prints it).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
n_duplicates = np.nan
chp_clean = chp.copy()
```
by
```python
n_duplicates = chp.duplicated().sum()
chp_clean = chp.drop_duplicates().copy()
```

In [ ]:
n_duplicates = np.nan                  # TODO: number of fully duplicated rows
chp_clean = chp.copy()                 # TODO: drop the duplicates
chp_clean["timestamp"] = pd.to_datetime(chp_clean["timestamp"])

print("duplicated rows:", n_duplicates)
print("rows after cleaning:", len(chp_clean), "| expected:", expected_rows)
print("each hour exactly once:", chp_clean["timestamp"].is_unique)

### ✏️ Your turn 2

**Technically correct, completely wrong.** `-999` is a *placeholder* for “no reading”, not a temperature.

1. Fill in the `TODO`: replace `-999` by `np.nan` in `ambient_c` (`.replace(-999, np.nan)`), and count how many
   values were replaced.
2. The cell prints the coldest and the warmest hour of the year and the mean ambient temperature.

**What to change** — edit the cell exactly like this, then run it:

Change `# TODO: replace the sentinel -999 by np.nan in chp_clean["ambient_c"]` into `chp_clean["ambient_c"] = chp_clean["ambient_c"].replace(-999, np.nan)`

In [ ]:
n_sentinels = (chp_clean["ambient_c"] == -999).sum()
# TODO: replace the sentinel -999 by np.nan in chp_clean["ambient_c"]

print("sentinel values:", n_sentinels)
print("min / max ambient:", chp_clean["ambient_c"].min(), chp_clean["ambient_c"].max())
print("mean ambient:", round(chp_clean["ambient_c"].mean(), 2))
print("missing ambient:", chp_clean["ambient_c"].isna().sum())

### ✏️ Your turn 3

**Physically impossible values.** The plant has an installed electrical capacity of **500 kW**, so no hourly average can
be higher.

1. Set `installed_kw = 500`.
2. Count the hours where `electric_kw > installed_kw`, set those values to `np.nan` (the usual default — we do not
   delete rows) and report the mean `electric_kw` **while the engine is running** (`engine_status == "running"`)
   before and after.

**What to change** — edit the cell exactly like this, then run it:

Change `installed_kw = 1000` into `installed_kw = 500`

In [ ]:
installed_kw = 1000     # TODO: the real installed capacity is 500 kW

too_high = chp_clean["electric_kw"] > installed_kw
mean_before = chp_clean.loc[chp_clean["engine_status"] == "running", "electric_kw"].mean()
chp_clean.loc[too_high, "electric_kw"] = np.nan
mean_after = chp_clean.loc[chp_clean["engine_status"] == "running", "electric_kw"].mean()

print("hours above capacity:", too_high.sum())
print("mean electric_kw (running) before:", round(mean_before, 2), "| after:", round(mean_after, 2))
print("max electric_kw now:", chp_clean["electric_kw"].max())
print("missing per column:")
print(chp_clean.isna().sum())

## 2. Splitting data — and why order matters

Neighbouring hours are nearly the same hour. If we split a time series *randomly*, almost every test hour has its
neighbours in the training set. The test is then far too easy, because the "new" hour was already seen.
Split along **whatever is new in production**: new hours → split by time.

### ✏️ Your turn 4

Split the cleaned CHP data **by time**: first 70 % of the hours for training, next 15 % for validation, last 15 % for
test. The starter uses a *random* split instead.

1. Replace the `TODO` lines so that `train`, `val`, `test` are the first, middle and last consecutive blocks of
   `chp_clean` (sorted by time). With `n = len(chp_clean)`, use `iloc[:n_train]`, `iloc[n_train:n_train + n_val]`, `iloc[n_train + n_val:]`
   where `n_train = int(0.70 * n)` and `n_val = int(0.15 * n)`.
2. The cell reports (a) the mean `ambient_c` of each part and (b) the **share of test hours whose previous hour is in the
   training set**. For a random split this share is about 0.7; for a time split it is almost 0.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
shuffled = chp_s.sample(frac=1, random_state=0)
train = shuffled.iloc[:n_train]
val = shuffled.iloc[n_train:n_train + n_val]
test = shuffled.iloc[n_train + n_val:]
```
by
```python
train = chp_s.iloc[:n_train]
val = chp_s.iloc[n_train:n_train + n_val]
test = chp_s.iloc[n_train + n_val:]
```

In [ ]:
chp_s = chp_clean.sort_values("timestamp").reset_index(drop=True)
n = len(chp_s)
n_train, n_val = int(0.70 * n), int(0.15 * n)

shuffled = chp_s.sample(frac=1, random_state=0)
train = shuffled.iloc[:n_train]                      # TODO: first n_train rows of chp_s (in time order)
val = shuffled.iloc[n_train:n_train + n_val]         # TODO: next n_val rows
test = shuffled.iloc[n_train + n_val:]               # TODO: the rest

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} rows={len(part):5d}  from {part['timestamp'].min():%Y-%m-%d} to {part['timestamp'].max():%Y-%m-%d}"
          f"  mean ambient={part['ambient_c'].mean():6.2f}")

train_idx = set(train.index)
prev_in_train = [(i - 1) in train_idx for i in test.index]
print("share of test hours whose previous hour is in the training set:", round(np.mean(prev_in_train), 3))

**Explore the training set only.** Choosing what to look at is already a modelling decision. From now on, summaries that
guide decisions are computed on `train` — not on the whole file.

### ✏️ Your turn 5

Compute from the **training set** (the one you just made) the mean `electric_kw` while the engine is running, and the
share of training hours with the engine `stopped` (3 decimals). Fill in the `TODO` lines.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
mean_kw_train = np.nan
stopped_share_train = np.nan
```
by
```python
mean_kw_train = train.loc[train["engine_status"] == "running", "electric_kw"].mean()
stopped_share_train = (train["engine_status"] == "stopped").mean()
```

In [ ]:
mean_kw_train = np.nan      # TODO: mean electric_kw of train rows with engine_status == "running"
stopped_share_train = np.nan    # TODO: share of train rows with engine_status == "stopped"

print("mean electric_kw while running (train):", round(mean_kw_train, 2))
print("share stopped (train):", round(stopped_share_train, 3))
print("share stopped (test) :", round((test["engine_status"] == "stopped").mean(), 3))

### Splitting by group, and stratified splitting

`transformer_inspections.csv`: **one row is one distribution transformer** (3 000 of them), belonging to one of 60
substations. `failed = 1` if the transformer failed in the following year (rare!).

* *A new substation in production* → **split by group**: all transformers of a substation stay on the same side.
* *A rare event* → **stratify**, so that the test set contains failures in the same proportion as the training set.

In [ ]:
tr = pd.read_csv(DATA_URL + "transformer_inspections.csv")
print(tr.shape, "| failure rate:", round(tr["failed"].mean(), 4), "| failures:", tr["failed"].sum())
tr.head()

### ✏️ Your turn 6

Use scikit-learn's `train_test_split` (only to split — no modelling) with a 20 % test set and `random_state=0`.

1. Run it as it is (a plain random split) and note the number of failures in the test set.
2. Add `stratify=tr["failed"]` in the `TODO` position and run again.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
tr_train, tr_test = train_test_split(tr, test_size=0.2, random_state=0)
```
by
```python
tr_train, tr_test = train_test_split(tr, test_size=0.2, random_state=0, stratify=tr["failed"])
```

In [ ]:
from sklearn.model_selection import train_test_split

tr_train, tr_test = train_test_split(tr, test_size=0.2, random_state=0)     # TODO: add stratify=tr["failed"]

print("test rows:", len(tr_test))
print("failures in test set:", tr_test["failed"].sum())
print("failure rate  train:", round(tr_train["failed"].mean(), 4), " test:", round(tr_test["failed"].mean(), 4))
print("substations present in both train and test:",
      len(set(tr_train["substation_id"]) & set(tr_test["substation_id"])))

### ✏️ Your turn 7

Now split **by substation**: the test set must contain *whole* substations that never appear in training.

The starter picks 15 test substations at random. Fill in the two `TODO`s: `tr_test_g` = rows of `tr` whose
`substation_id` is in `test_subs`; `tr_train_g` = all other rows (use `~tr["substation_id"].isin(test_subs)`).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
tr_test_g = tr.iloc[0:0]
tr_train_g = tr.iloc[0:0]
```
by
```python
tr_test_g = tr[tr["substation_id"].isin(test_subs)]
tr_train_g = tr[~tr["substation_id"].isin(test_subs)]
```

In [ ]:
rng = np.random.default_rng(0)
subs = np.sort(tr["substation_id"].unique())
test_subs = rng.choice(subs, size=15, replace=False)

tr_test_g = tr.iloc[0:0]     # TODO: rows in test_subs
tr_train_g = tr.iloc[0:0]    # TODO: all other rows

print("test substations:", len(test_subs))
print("train rows:", len(tr_train_g), "| test rows:", len(tr_test_g))
print("substations in both:", len(set(tr_train_g["substation_id"]) & set(tr_test_g["substation_id"])))
print("failures in test:", tr_test_g["failed"].sum(), "| failure rate test:", round(tr_test_g["failed"].mean(), 4) if len(tr_test_g) else None)

## Summary

* Run **shape → dtypes → describe → isna** on every new dataset; check that the count adds up.
* Placeholders (`-999`), duplicates and physically impossible values are *technically correct, completely wrong*.
* Split by time for time series, by group when new groups appear, and stratify for rare events. Explore **only the training set**.